# 15 · Bảng 2×2 OCR (train × test) + full test 10.028 câu (W9)

**Vấn đề 1 — lợi ích của B3 đến từ đâu?** Phần A (notebook 14) cho thấy B3 phụ thuộc OCR. Nhưng còn thiếu ô *"train không OCR, test có OCR"* trên cùng tập con, nên chưa tách được: lợi ích đến từ **train cùng OCR**, hay chỉ cần **có OCR lúc test**?

| | Test không OCR | Test có OCR |
|---|---|---|
| Train không OCR | c00 = B2-sub (W08) | **c01 = B2-sub + OCR (chạy ở đây)** |
| Train có OCR | c10 = B3 bỏ OCR (W09) | c11 = B3 (W08) |

**Vấn đề 2 — test2000 chỉ là tập con.** So sánh chính B2-sub → B3 và so với ViTextBLIP-2 cần chạy trên đúng full test 10.028 câu mà paper dùng.

**Luật ghi trước:** `docs/W09/README.md` (mục *Luật ô 2×2* và *Luật full test*).

| Phần | Việc | Thời gian (L4) |
|---|---|---|
| A | Eval B2-sub seed 42 **có OCR** trên test2000 → bảng 2×2 | ≈ 40 phút |
| B | B3 + B2-sub seed 42 trên **full test** (cần notebook 10 Phần C chạy xong trước) | ≈ 5,7 giờ |

**Thứ tự trong ngày:** notebook 10 **Phần C** (T4, OCR ảnh full test còn thiếu) → notebook này **Run all** (L4). Nếu chạy notebook này trước khi có OCR full test, Phần A vẫn chạy xong rồi notebook **dừng có chủ đích** ở mục 5.

## 1. Thiết lập môi trường

Giống notebook 13/14 (cùng pin `transformers==5.17.0`). Dừng hẳn nếu GPU không phải L4.

*Kỳ vọng: commit mới nhất (có notebook 15), `GPU: NVIDIA L4`, `✅ Thiết lập xong`.*

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')
assert os.path.exists('notebooks/15_ocr_factorial_fulltest_qwen25vl.ipynb'), 'Code cũ -> git pull chưa lấy được commit notebook 15'

!pip install -q transformers==5.17.0 accelerate qwen-vl-utils bitsandbytes peft huggingface_hub pyyaml

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # không in token ra
    print('HF_TOKEN: đã nạp')
except Exception:
    print('HF_TOKEN: không có (ổn nếu zip ảnh đã nằm trên Drive)')

import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU'
print('GPU:', GPU)
if 'L4' not in GPU:
    raise SystemExit('❌ Phải chạy trên L4 (cùng GPU với B2-sub/B3). Runtime -> Change runtime type -> L4, rồi chạy lại cell này.')
print('✅ Thiết lập xong')

## 2. Tên run + kiểm tra trước khi tốn GPU

- Cấu hình OCR đọc từ `ocr_selection.json` (W08 bước 5), **không gõ tay**.
- Adapter B2-sub và B3 seed 42 trên Drive phải **đúng sha256** đã ghi trong `run_signature.json` của eval W08 (đúng adapter đã chấm).
- OCR cache: test2000 phải đủ (Phần A). Full test: chỉ báo số ảnh đã có — thiếu thì Phần B dừng ở mục 5.
- Selftest: `factorial_2x2`, `run_baseline`, `paper_consistency`.

*Kỳ vọng: `paddledet_vietocr | v1 | 600 | 0.0`, 2 dòng `adapter … sha256 khớp ✅`, `test2000 1486/1486`, dòng `full test …/3353` (đủ hay thiếu đều được ở bước này), `✅ selftest OK (3 script)`.*

In [ ]:
import json, subprocess
import pandas as pd
from src.data.ocr import default_cache_path, load_ocr_cache, images_of
from src.eval.run_baseline import prepare_images, adapter_fingerprint, DATASETS

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
DS = 'vitextvqa_official'
EXP_ROOT = f'{DRIVE_ROOT}/experiments'
ADP_ROOT = f'{DRIVE_ROOT}/adapters'
TEST2000 = f"{DRIVE_ROOT}/{DATASETS[DS]['data_rel']}"
FULL_TEST = f'{DRIVE_ROOT}/data/{DS}/test.json'
DIR_F = f'{EXP_ROOT}/W09_ocr_factorial_2x2_test2000_L4_seed42'
DIR_FULL = f'{EXP_ROOT}/W09_fulltest_B3_vs_B2sub_L4_seed42'
CHUNK = 500
RUN_B2SUB_FULL = True     # B2-sub full test (≈ 2,8 giờ) — cần cho so sánh chính lặp lại trên full test

sel = json.load(open('experiments/W08_ocr_select_dev300_L4_seed42/ocr_selection.json', encoding='utf-8'))['decision']
BACKEND, TPL, MAXC, MINC = sel['backend'], sel['template'], sel['max_chars'], sel['min_conf']
print(BACKEND, '|', TPL, '|', MAXC, '|', MINC)
OCR_ARGS = f'--mode ocr --ocr_backend {BACKEND} --ocr_template {TPL} --ocr_max_chars {MAXC} --ocr_min_conf {MINC}'

TAIL = 'qwen25vl_vitextvqa_sub8k_ep1_bf16_L4_seed42'
ADAPTER = {'B2sub': f'{ADP_ROOT}/W08_B2sub_qlora_r16_{TAIL}/epoch_1',
           'B3':    f'{ADP_ROOT}/W08_B3_qlora_ocr_vietocr_v1_r16_{TAIL}/epoch_1'}
CELL = {'c00': 'W08_B2sub_eval_qlora_r16_sub8k_ep1_vitextvqa_test2000_bf16_L4_seed42',
        'c01': 'W09_B2subocr_eval_qlora_r16_sub8k_ep1_vitextvqa_test2000_bf16_L4_seed42',
        'c10': 'W09_B3noocr_eval_qlora_ocr_vietocr_v1_r16_sub8k_ep1_vitextvqa_test2000_bf16_L4_seed42',
        'c11': 'W08_B3_eval_qlora_ocr_vietocr_v1_r16_sub8k_ep1_vitextvqa_test2000_bf16_L4_seed42'}
FULL = {'B3':    {'run': 'W09_B3_eval_qlora_ocr_vietocr_v1_r16_sub8k_ep1_vitextvqa_testfull_bf16_L4_seed42',
                  'adapter': ADAPTER['B3'], 'extra': OCR_ARGS, 'subset_run': CELL['c11']},
        'B2sub': {'run': 'W09_B2sub_eval_qlora_r16_sub8k_ep1_vitextvqa_testfull_bf16_L4_seed42',
                  'adapter': ADAPTER['B2sub'], 'extra': '', 'subset_run': CELL['c00']}}
B1_FULL = 'W07_B1_zeroshot_qwen25vl_vitextvqa_testfull_bf16_L4_seed42'
B2_FULL = 'W07_B2_eval_qlora_r16_ep1_vitextvqa_testfull_bf16_L4_seed42'

# Adapter trên Drive phải đúng adapter đã chấm ở W08 (sha256 trong run_signature.json đã lên git)
for k, run in (('B2sub', CELL['c00']), ('B3', CELL['c11'])):
    want = json.load(open(f'experiments/{run}/run_signature.json', encoding='utf-8'))['adapter']['weights_sha256']
    got = adapter_fingerprint(ADAPTER[k])['weights_sha256']
    assert got == want, f'adapter {k} trên Drive KHÁC adapter đã chấm ({got[:12]} ≠ {want[:12]}) -> dừng, báo Claude'
    print(f'adapter {k} sha256 khớp ✅ ({got[:12]})')

IMG = prepare_images(DS, DRIVE_ROOT)
cache = load_ocr_cache(default_cache_path(DRIVE_ROOT, DS, BACKEND))
have = set(cache['images'])
t2k, tfull = set(images_of([TEST2000], IMG)), set(images_of([FULL_TEST], IMG))
FULL_OCR_READY = tfull <= have
print(f"test2000 {len(t2k & have)}/{len(t2k)} | full test {len(tfull & have)}/{len(tfull)} -> "
      f"{'đủ ✅' if FULL_OCR_READY else 'CHƯA đủ: Phần B sẽ dừng ở mục 5 (chạy notebook 10 Phần C trước)'}")
assert t2k <= have, 'OCR cache thiếu ảnh test2000 -> dừng, báo Claude'

def run_cmd(cmd):
    # Chạy lệnh, in log trực tiếp; lỗi -> DỪNG notebook
    print('$', cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'Lệnh lỗi (mã {p.returncode}) -> xem log ở trên')

outs = [subprocess.run(f'python -m {m} --selftest', shell=True, capture_output=True, text=True).stdout
        for m in ('src.eval.factorial_2x2', 'src.eval.run_baseline', 'src.eval.paper_consistency')]
assert all('SELFTEST OK' in o for o in outs), outs
print('✅ selftest OK (3 script)')

## 3. Phần A — Ô c01: B2-sub **có OCR** lúc test (≈ 40 phút)

Adapter B2-sub seed 42 (train **không** OCR), prompt **có** dòng OCR — đúng cấu hình OCR đóng băng như B3.

*Kỳ vọng: dòng `OCR: … 99.85% mẫu có chữ`, 4 lô, `== KẾT QUẢ: EM=…`.*

In [ ]:
if os.path.exists(f"{EXP_ROOT}/{CELL['c01']}/metrics.json"):
    print('⏭ c01: đã có -> bỏ qua')
else:
    run_cmd(f"python -u -m src.eval.run_baseline --dataset {DS} --run_name {CELL['c01']} --adapter {ADAPTER['B2sub']} {OCR_ARGS}")

## 4. Phần A — Bảng 2×2 và kết luận theo luật

`factorial_2x2.py`: EM 4 ô, 4 hiệu ứng đơn (McNemar + ΔANLS CI95), tương tác theo câu (CI95 bootstrap ghép cặp).

| Kết luận | Điều kiện (K = c01 → c11) |
|---|---|
| TƯƠNG TÁC: CẦN OCR Ở CẢ TRAIN LẪN TEST | K: EM tăng & p < 0,05 & cận dưới CI95 tương tác EM > 0 |
| CỘNG GỘP: TRAIN CÙNG OCR CÓ LỢI RIÊNG | K: EM tăng & p < 0,05, CI95 tương tác chứa 0 |
| OCR LÚC TEST LÀ ĐỦ | K: p ≥ 0,05 |
| TRAIN CÙNG OCR KÉM HƠN | K: EM giảm & p < 0,05 |

*Kỳ vọng: bảng 2×2, 4 dòng hiệu ứng, 2 dòng tương tác, dòng `➜ KẾT LUẬN 2×2 THEO LUẬT: …`, file `factorial_summary.json`. **Điểm dừng 1** — gửi output mục này cho Claude.*

In [ ]:
os.makedirs(DIR_F, exist_ok=True)
args = ' '.join(f'--{c} {EXP_ROOT}/{r}' for c, r in CELL.items())
run_cmd(f'python -m src.eval.factorial_2x2 {args} --out {DIR_F}/factorial_summary.json')

## 5. Phần B — Full test 10.028 câu: B3 rồi B2-sub (≈ 5,7 giờ)

Cùng adapter, cùng cấu hình như test2000; chỉ đổi file dữ liệu sang `test.json`. Mỗi lô 500 câu ghi lên Drive ngay khi xong — bị ngắt thì Run all lại, lô đã xong được bỏ qua (mất tối đa ≈ 10 phút).

Nếu OCR cache **chưa đủ** ảnh full test, cell này **dừng có chủ đích**: chạy notebook 10 Phần C (T4), rồi mở lại notebook này và Run all (Phần A đã xong sẽ được bỏ qua).

*Kỳ vọng: B3 có dòng `OCR: …`, 21 lô, `== KẾT QUẢ`; rồi B2-sub 21 lô, `== KẾT QUẢ`.*

In [ ]:
if not FULL_OCR_READY:
    raise SystemExit('⏸ OCR cache chưa đủ ảnh full test -> chạy notebook 10 Phần C (T4) trước, rồi Run all lại notebook này.')
for k, r in FULL.items():
    if k == 'B2sub' and not RUN_B2SUB_FULL:
        print('B2-sub full test: tắt -> bỏ qua'); continue
    if os.path.exists(f"{EXP_ROOT}/{r['run']}/metrics.json"):
        print(f'⏭ {k} full test: đã có -> bỏ qua'); continue
    run_cmd(f"python -u -m src.eval.run_baseline --dataset {DS} --data_file {FULL_TEST} --run_name {r['run']} "
            f"--chunk {CHUNK} --adapter {r['adapter']} {r['extra']}")

## 6. Phần B — Kết quả full test theo luật

1. **Tái lập:** trên 2.000 câu chung, dự đoán run full trùng ≥ 99% với run test2000 (W08). Không đạt → dừng, báo Claude.
2. **So sánh chính lặp lại:** B2-sub → B3 trên full test (EM tăng & p < 0,05 → TÁI LẬP TRÊN FULL TEST).
3. **So với ViTextBLIP-2** (Table 6: EM 25,48 · F1 53,95): "cao hơn" chỉ khi cận dưới CI95 > số công bố.
4. **Chỉ báo cáo:** B2 full train (W07) → B3.

*Kỳ vọng: 2 dòng tái lập ✅, bảng full test (B1, B2, B2-sub, B3, ViTextBLIP-2), bảng so với ViTextBLIP-2, dòng `➜ SO SÁNH CHÍNH TRÊN FULL TEST: …`, file `fulltest_summary.json`. **Điểm dừng 2.***

In [ ]:
from src.eval import paper_consistency as pc
from src.eval.metrics import exact_match
os.makedirs(DIR_FULL, exist_ok=True)
def load(run):
    d = f'{EXP_ROOT}/{run}'
    return (json.load(open(f'{d}/metrics.json', encoding='utf-8')), json.load(open(f'{d}/predictions.json', encoding='utf-8')))

M, P, DET = {}, {}, {}
for k, r in FULL.items():
    if not os.path.exists(f"{EXP_ROOT}/{r['run']}/metrics.json"):
        print(f'{k}: chưa có kết quả full test'); continue
    M[k], P[k] = load(r['run'])
    sub = json.load(open(f"{EXP_ROOT}/{r['subset_run']}/predictions.json", encoding='utf-8'))
    by_id = {p['question_id']: p for p in P[k]}
    same = [by_id[s['question_id']]['prediction'] == s['prediction'] for s in sub]
    DET[k] = {'n': len(same), 'identical_pct': 100 * sum(same) / len(same),
              'EM_subset_run': 100 * sum(exact_match(s['prediction'], s['answers']) for s in sub) / len(sub)}
    DET[k]['pass'] = DET[k]['identical_pct'] >= 99
    print(f"{k}: trùng {DET[k]['identical_pct']:.2f}% với run test2000 -> {'✅' if DET[k]['pass'] else '❌'}")
if not all(d['pass'] for d in DET.values()):
    raise SystemExit('❌ Run full không tái lập run test2000 (< 99% trùng) -> KHÔNG dùng số liệu; gửi output cho Claude.')

rows = []
for label, run in (('B1 zero-shot (W07)', B1_FULL), ('B2 full train (W07)', B2_FULL)):
    if os.path.exists(f'{EXP_ROOT}/{run}/metrics.json'):
        mm = json.load(open(f'{EXP_ROOT}/{run}/metrics.json', encoding='utf-8'))['metrics']
        rows.append({'run': label, 'EM': 100 * mm['exact_match'], 'ANLS': 100 * mm['anls'], 'F1 (phụ)': 100 * mm['token_f1']})
for k in P:
    mm = M[k]['metrics']
    rows.append({'run': {'B3': 'B3 (8k + OCR)', 'B2sub': 'B2-sub (8k, không OCR)'}[k], 'EM': 100 * mm['exact_match'],
                 'ANLS': 100 * mm['anls'], 'F1 (phụ)': 100 * mm['token_f1'], 's/câu': M[k]['meta']['sec_per_sample'],
                 'giờ suy luận': round(M[k]['meta']['inference_time_sec'] / 3600, 2)})
rows.append({'run': 'ViTextBLIP-2 (paper, Table 6)', 'EM': pc.TABLE6_VITEXTBLIP2['em'], 'F1 (phụ)': pc.TABLE6_VITEXTBLIP2['f1']})
print('==== ViTextVQA FULL test (10.028 câu) ====')
display(pd.DataFrame(rows).round(2))

SOTA = {k: pc.vitextblip2_report(P[k]) for k in P}
display(pd.DataFrame([{'run': k, 'metric': met, 'nhóm': SOTA[k][met]['ours_pct'], 'CI95': SOTA[k][met]['ours_ci95_pct'],
                       'paper': SOTA[k][met]['published_pct'], 'kết luận': SOTA[k][met]['verdict']}
                      for k in SOTA for met in ('EM', 'F1')]))

def paired(run_a, run_b, la, lb, out):
    r = subprocess.run(f'python src/eval/compare_runs.py --run_a {EXP_ROOT}/{run_a} --run_b {EXP_ROOT}/{run_b} '
                       f'--label_a {la} --label_b {lb} --out {out}', shell=True, capture_output=True, text=True)
    assert r.returncode == 0, r.stdout + r.stderr
    p = json.load(open(out, encoding='utf-8')); p.pop('only_b_question_ids', None)
    print(f"{la:>6} -> {lb:<4} EM {p['em_a']:.4f} -> {p['em_b']:.4f} | chỉ {la} đúng {p['n_only_a']}, chỉ {lb} đúng {p['n_only_b']}, "
          f"p={p['p_value']:.3g} | ΔANLS {p['anls_diff']:+.4f} CI95 [{p['anls_ci95'][0]:+.4f}; {p['anls_ci95'][1]:+.4f}]")
    return p

PAIRS, verdict = {}, None
if 'B3' in P and 'B2sub' in P:
    PAIRS['B2sub_vs_B3_full'] = p = paired(FULL['B2sub']['run'], FULL['B3']['run'], 'B2sub', 'B3', f'{DIR_FULL}/paired_B2sub_vs_B3_full.json')
    verdict = ('TÁI LẬP TRÊN FULL TEST' if p['em_b'] > p['em_a'] and p['p_value'] < 0.05 else
               'KÉM HƠN có ý nghĩa' if p['em_b'] < p['em_a'] and p['p_value'] < 0.05 else 'KHÔNG KHÁC BIỆT có ý nghĩa')
    print(f"➜ SO SÁNH CHÍNH TRÊN FULL TEST (B2-sub -> B3): {verdict}")
if 'B3' in P and os.path.exists(f'{EXP_ROOT}/{B2_FULL}/predictions.json'):
    PAIRS['B2_vs_B3_full_info'] = paired(B2_FULL, FULL['B3']['run'], 'B2', 'B3', f'{DIR_FULL}/paired_B2_vs_B3_full_info.json')

json.dump({'what': 'W09 Phần B: B3 và B2-sub seed 42 trên ViTextVQA full test 10.028 câu',
           'rule': 'docs/W09/README.md mục Luật full test', 'ocr_config': sel, 'table': rows,
           'determinism_vs_test2000': DET, 'vs_vitextblip2': SOTA, 'reference': pc.TABLE6_VITEXTBLIP2,
           'source': pc.PAPER_SOURCE, 'paired': PAIRS, 'verdict_main_full': verdict, 'gpu': GPU},
          open(f'{DIR_FULL}/fulltest_summary.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2, default=str)
print('->', f'{DIR_FULL}/fulltest_summary.json')

## 7. ⏸ DỪNG

Gửi Claude output **mục 4** (sau Phần A) và **mục 6** (sau Phần B). Sau đó **Runtime → Disconnect and delete runtime**.